# Evaluate DAPO Checkpoint

Restore the selected 1.5B SFT and DAPO checkpoints, evaluate both on GSM8K test, then run paired comparison and error analysis.

## 1. Pull Repo / Setup

In [ ]:
import importlib.util
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
REPO_DIR = Path("/content/strategy-distill-rl")
IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    if REPO_DIR.exists():
        print(f"Pulling latest repo in {REPO_DIR}")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    else:
        print(f"Cloning repo to {REPO_DIR}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"Local run detected. Current directory: {Path.cwd()}")

PROJECT_ROOT = Path.cwd()
print("Working directory:", PROJECT_ROOT)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


## 2. Install Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

AUTO_INSTALL_MISSING_DEPENDENCIES = True
REQUIRED_PACKAGES = {
    "peft": "peft",
    "accelerate": "accelerate",
    "transformers": "transformers",
    "tqdm": "tqdm",
    "pandas": "pandas",
}

missing = [pkg for import_name, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing:
    if not AUTO_INSTALL_MISSING_DEPENDENCIES:
        raise ModuleNotFoundError("Missing packages: " + ", ".join(missing))
    print("Installing missing dependencies:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
else:
    print("All dependencies are available.")

# Older Colab torchao builds can break PEFT import. This project does not need torchao.
try:
    import torchao
    version = getattr(torchao, "__version__", "0.0.0")
    major_minor = tuple(int(part) for part in version.split(".")[:2] if part.isdigit())
    if major_minor and major_minor < (0, 16):
        print(f"Uninstalling incompatible torchao {version}")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
except Exception as exc:
    print("torchao check skipped:", exc)


## 3. Config

In [ ]:
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/RL/Data")
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/RL/Checkpoints")
DRIVE_REPORT_DIR = Path("/content/drive/MyDrive/RL/Reports")

SFT_ZIP = DRIVE_CHECKPOINT_DIR / "balanced_r16_a32_4000_20260630_233638.zip"
DAPO_ZIP = DRIVE_CHECKPOINT_DIR / "dapo_1p5b_g8_lr5e7_e1_20260704_164638.zip"

SFT_ADAPTER_PATH = Path("checkpoints/student_sft/balanced_r16_a32_4000")
DAPO_ADAPTER_PATH = Path("checkpoints/student_dapo/dapo_1p5b_g8_lr5e7_e1")
TEST_PATH = Path("data/gsm8k_clean_test.jsonl")
DRIVE_TEST_PATH = DRIVE_DATA_DIR / "gsm8k_clean_test.jsonl"

RUN_DIR = Path("runs/dapo_checkpoint_eval")
RUN_DIR.mkdir(parents=True, exist_ok=True)

EVAL_NUM_SAMPLES = -1
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 512
MAX_PRINT = 30

print("SFT zip:", SFT_ZIP)
print("DAPO zip:", DAPO_ZIP)
print("Run dir:", RUN_DIR)


## 4. Helpers

In [ ]:
import json
import shutil
import subprocess
import zipfile
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd


def mount_drive_if_needed():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive mount skipped or unavailable:", exc)


def run_command(args):
    command = [str(arg) for arg in args]
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    output_lines = []
    for line in process.stdout:
        print(line, end="")
        output_lines.append(line)
    return_code = process.wait()
    if return_code != 0:
        tail = "".join(output_lines[-80:])
        raise RuntimeError(
            f"Command failed with exit code {return_code}: {' '.join(command)}\n"
            f"Last output lines:\n{tail}"
        )


def restore_zip_checkpoint(zip_path, target_dir, label):
    mount_drive_if_needed()
    zip_path = Path(zip_path)
    target_dir = Path(target_dir)
    if not zip_path.exists():
        raise FileNotFoundError(f"Missing {label} checkpoint zip: {zip_path}")
    if target_dir.exists():
        print(f"{label} checkpoint already restored: {target_dir}")
        return
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f"Restoring {label} checkpoint from {zip_path} to {target_dir}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(target_dir)


def copy_from_drive_if_missing(local_path, drive_path, label):
    mount_drive_if_needed()
    local_path = Path(local_path)
    drive_path = Path(drive_path)
    if local_path.exists():
        print(f"{label} already exists: {local_path}")
        return
    if not drive_path.exists():
        raise FileNotFoundError(f"Missing {label} in Drive: {drive_path}")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, local_path)
    print(f"Copied {label}: {drive_path} -> {local_path}")


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def evaluate_adapter(run_name, adapter_path):
    output_path = RUN_DIR / f"{run_name}_outputs.jsonl"
    metrics_path = RUN_DIR / f"{run_name}_metrics.json"
    run_command([
        "python", "-B", "scripts/evaluate_student.py",
        "--model-name", MODEL_NAME,
        "--adapter-path", adapter_path,
        "--input-path", TEST_PATH,
        "--output-path", output_path,
        "--metrics-path", metrics_path,
        "--num-samples", EVAL_NUM_SAMPLES,
        "--batch-size", EVAL_BATCH_SIZE,
        "--max-new-tokens", EVAL_MAX_NEW_TOKENS,
    ])
    row = read_json(metrics_path)
    row["run"] = run_name
    row["metrics_path"] = str(metrics_path)
    row["output_path"] = str(output_path)
    return row


def by_id(rows):
    return {row["id"]: row for row in rows}


def paired_buckets(sft_rows, dapo_rows):
    sft = by_id(sft_rows)
    dapo = by_id(dapo_rows)
    common_ids = sorted(set(sft) & set(dapo))
    return {
        "common_ids": common_ids,
        "fixes": [(sft[i], dapo[i]) for i in common_ids if sft[i].get("is_correct") != 1 and dapo[i].get("is_correct") == 1],
        "regressions": [(sft[i], dapo[i]) for i in common_ids if sft[i].get("is_correct") == 1 and dapo[i].get("is_correct") != 1],
        "both_wrong": [(sft[i], dapo[i]) for i in common_ids if sft[i].get("is_correct") != 1 and dapo[i].get("is_correct") != 1],
        "both_correct": [(sft[i], dapo[i]) for i in common_ids if sft[i].get("is_correct") == 1 and dapo[i].get("is_correct") == 1],
    }


def print_pair(sft_row, dapo_row, title):
    print("=" * 120)
    print(title)
    print("id:", dapo_row.get("id"))
    print("question:", dapo_row.get("question"))
    print("ground_truth:", dapo_row.get("ground_truth"))
    print("\n--- SFT ---")
    print("answer:", sft_row.get("model_answer"), "correct:", sft_row.get("is_correct"), "format:", sft_row.get("is_format_valid"))
    print(sft_row.get("model_output"))
    print("\n--- DAPO ---")
    print("answer:", dapo_row.get("model_answer"), "correct:", dapo_row.get("is_correct"), "format:", dapo_row.get("is_format_valid"))
    print(dapo_row.get("model_output"))


def print_rows(rows, title, max_print=MAX_PRINT):
    print("#" * 120)
    print(title)
    print("#" * 120)
    for idx, row in enumerate(rows[:max_print], start=1):
        print("=" * 100)
        print(f"sample #{idx} | id={row.get('id')}")
        print("question:", row.get("question"))
        print("ground_truth:", row.get("ground_truth"))
        print("model_answer:", row.get("model_answer"))
        print("is_correct:", row.get("is_correct"), "is_format_valid:", row.get("is_format_valid"), "is_usable:", row.get("is_usable"))
        print("format_checks:", row.get("format_checks"))
        print("\nmodel_output:")
        print(row.get("model_output"))
        print("\nraw_model_output:")
        print(row.get("raw_model_output"))


## 5. Restore Checkpoints and Data

In [ ]:
restore_zip_checkpoint(SFT_ZIP, SFT_ADAPTER_PATH, "SFT")
restore_zip_checkpoint(DAPO_ZIP, DAPO_ADAPTER_PATH, "DAPO")
copy_from_drive_if_missing(TEST_PATH, DRIVE_TEST_PATH, "GSM8K test data")


## 6. Evaluate SFT vs DAPO

In [ ]:
rows = []
rows.append(evaluate_adapter("sft_baseline", SFT_ADAPTER_PATH))
rows.append(evaluate_adapter("dapo", DAPO_ADAPTER_PATH))

metrics_df = pd.DataFrame(rows)
metric_cols = [
    "run", "total", "accuracy", "loose_math_accuracy", "format_valid_rate", "usable_rate",
    "correct", "loose_correct", "format_valid", "usable", "metrics_path", "output_path",
]
display(metrics_df[metric_cols])

sft = metrics_df[metrics_df["run"] == "sft_baseline"].iloc[0]
dapo = metrics_df[metrics_df["run"] == "dapo"].iloc[0]
delta = {
    "accuracy_delta": dapo["accuracy"] - sft["accuracy"],
    "loose_math_accuracy_delta": dapo["loose_math_accuracy"] - sft["loose_math_accuracy"],
    "format_valid_rate_delta": dapo["format_valid_rate"] - sft["format_valid_rate"],
    "usable_rate_delta": dapo["usable_rate"] - sft["usable_rate"],
    "correct_delta": dapo["correct"] - sft["correct"],
    "usable_delta": dapo["usable"] - sft["usable"],
}
delta_df = pd.DataFrame([delta])
display(delta_df)

metrics_df.to_csv(RUN_DIR / "sft_vs_dapo_metrics.csv", index=False)
delta_df.to_csv(RUN_DIR / "sft_vs_dapo_delta.csv", index=False)
print("Saved comparison CSVs to", RUN_DIR)


## 7. Paired Fix / Regression Analysis

In [ ]:
sft_rows = read_jsonl(RUN_DIR / "sft_baseline_outputs.jsonl")
dapo_rows = read_jsonl(RUN_DIR / "dapo_outputs.jsonl")
buckets = paired_buckets(sft_rows, dapo_rows)

summary = {
    "common": len(buckets["common_ids"]),
    "fixes_sft_wrong_dapo_correct": len(buckets["fixes"]),
    "regressions_sft_correct_dapo_wrong": len(buckets["regressions"]),
    "both_wrong": len(buckets["both_wrong"]),
    "both_correct": len(buckets["both_correct"]),
    "net_gain": len(buckets["fixes"]) - len(buckets["regressions"]),
}
display(pd.DataFrame([summary]))
write_json(summary, RUN_DIR / "paired_summary.json")


## 8. Error and Strategy Summaries

In [ ]:
dapo_wrong = [row for row in dapo_rows if row.get("is_correct") != 1]
format_invalid = [row for row in dapo_rows if row.get("is_format_valid") != 1]

strategy_rows = []
for name, rows_ in [("all", dapo_rows), ("wrong", dapo_wrong), ("format_invalid", format_invalid)]:
    counts = Counter(row.get("strategy") or "missing" for row in rows_)
    for strategy, count in counts.most_common():
        strategy_rows.append({"bucket": name, "strategy": strategy, "count": count})

display(pd.DataFrame(strategy_rows))
print("DAPO wrong:", len(dapo_wrong))
print("DAPO format invalid:", len(format_invalid))


## 9. Print Concrete Cases

In [ ]:
for sft_row, dapo_row in buckets["fixes"][:MAX_PRINT]:
    print_pair(sft_row, dapo_row, "FIX: SFT wrong -> DAPO correct")

for sft_row, dapo_row in buckets["regressions"][:MAX_PRINT]:
    print_pair(sft_row, dapo_row, "REGRESSION: SFT correct -> DAPO wrong")

print_rows(dapo_wrong, "DAPO WRONG OUTPUTS", max_print=MAX_PRINT)


## 10. Save Report Artifacts to Drive

In [ ]:
SAVE_REPORTS_TO_DRIVE = True

if SAVE_REPORTS_TO_DRIVE:
    mount_drive_if_needed()
    DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_base = Path("/tmp") / f"dapo_checkpoint_eval_{timestamp}"
    archive_path = shutil.make_archive(str(archive_base), "zip", RUN_DIR)
    target = DRIVE_REPORT_DIR / Path(archive_path).name
    shutil.copy2(archive_path, target)
    print("Saved report archive to:", target)
else:
    print("SAVE_REPORTS_TO_DRIVE=False; skipping archive.")
